# 04 — Run MEFISTO (Python)

MEFISTO is fit to the pseudocount-free rCLR matrices created in notebook 02. Original count zeros remain missing (`NaN`) in these matrices and are passed to MEFISTO as missing values.


In [1]:
from datetime import datetime
from pathlib import Path
import time
import anndata as ad
import muon as mu
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

BATCHES_TO_RUN = []
NUMBER_OF_FACTORS = 5
NUMBER_OF_ITERATIONS = 1000
RIDGE = 1e-5
RANDOM_SEED = 20260802

root = Path(".") if Path("data").exists() else Path("..")
split_folder = sorted(
    path for path in (root / "data" / "splits").iterdir()
    if path.is_dir() and any(path.glob("batch_*/train_mefisto_rclr.csv.gz"))
)[-1]
output_folder = root / "data" / "mefisto" / datetime.now().strftime("%Y%m%d_%H%M%S")
output_folder.mkdir(parents=True)

batch_folders = sorted(path for path in split_folder.glob("batch_*") if path.is_dir())
if BATCHES_TO_RUN:
    batch_folders = [x for x in batch_folders if x.name in BATCHES_TO_RUN]


/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/venv/lib/python3.10/site-packages/muon/_core/preproc.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
def summarize(samples):
    factors = [c for c in samples if c.startswith("factor_")]
    rows = []
    for subject, g in samples.groupby("subject_id"):
        g = g.sort_values("time")
        t = g["time"].to_numpy(float)
        t = (t - t.min()) / max(t.max() - t.min(), 1)
        row = {"subject_id": subject, "label": str(g["label"].iloc[0])}
        for factor in factors:
            y = g[factor].to_numpy(float)
            row[f"{factor}_mean"] = y.mean()
            row[f"{factor}_slope"] = np.polyfit(t, y, 1)[0]
        rows.append(row)
    return pd.DataFrame(rows)

def project(values, weights):
    out = []
    for row in values.to_numpy(float):
        keep = np.isfinite(row)
        W = weights[keep]
        out.append(row[keep] @ W @ np.linalg.pinv(W.T @ W + RIDGE * np.eye(W.shape[1])))
    return np.vstack(out)


In [ ]:
def run_batch(folder, number):
    started = time.perf_counter()
    batch_output = output_folder / folder.name
    batch_output.mkdir()

    train = pd.read_csv(folder / "train_mefisto_rclr.csv.gz", dtype={"sample_id": str}).set_index("sample_id")
    test = pd.read_csv(folder / "test_mefisto_rclr.csv.gz", dtype={"sample_id": str}).set_index("sample_id")
    train_meta = pd.read_csv(folder / "train_metadata.csv.gz",
                             dtype={"sample_id": str, "subject_id": str, "label": str})
    test_meta = pd.read_csv(folder / "test_metadata.csv.gz",
                            dtype={"sample_id": str, "subject_id": str, "label": str})
    train = train.loc[train_meta["sample_id"]]
    test = test.loc[test_meta["sample_id"], train.columns]

    mean = train.mean()
    scale = train.std(ddof=0).replace(0, 1)
    train = (train - mean) / scale
    test = (test - mean) / scale

    obs = train_meta.set_index("sample_id")[["subject_id", "time", "label"]].copy()
    obs["model_group"] = pd.factorize(obs["subject_id"])[0].astype(str)
    adata = ad.AnnData(train.loc[obs.index].to_numpy(np.float32), obs=obs,
                       var=pd.DataFrame(index=train.columns.astype(str)))

    mu.tl.mofa(
        adata, groups_label="model_group", likelihoods="gaussian",
        n_factors=NUMBER_OF_FACTORS, n_iterations=NUMBER_OF_ITERATIONS,
        convergence_mode="medium", smooth_covariate="time",
        smooth_kwargs={"scale_cov": True, "sparseGP": False, "model_groups": False},
        seed=RANDOM_SEED + number, outfile=str(batch_output / "model.hdf5"),
        quiet=True, copy=False
    )

    train_factors = np.asarray(adata.obsm["X_mofa"], float)
    weights = np.asarray(adata.varm["LFs"], float)
    names = [f"factor_{i+1}" for i in range(train_factors.shape[1])]

    train_projection = project(train, weights)
    test_projection = project(test, weights)
    calibration = np.linalg.lstsq(
        np.column_stack([np.ones(len(train_projection)), train_projection]),
        train_factors, rcond=None
    )[0]
    test_factors = np.column_stack([np.ones(len(test_projection)), test_projection]) @ calibration

    train_samples = pd.DataFrame(train_factors, index=obs.index, columns=names).rename_axis("sample_id").reset_index()
    test_samples = pd.DataFrame(test_factors, index=test_meta["sample_id"], columns=names).rename_axis("sample_id").reset_index()
    train_samples = train_samples.merge(train_meta, on="sample_id")
    test_samples = test_samples.merge(test_meta, on="sample_id")

    train_scores, test_scores = summarize(train_samples), summarize(test_samples)
    score_columns = [c for c in train_scores if c not in ["subject_id", "label"]]
    classifier = LogisticRegression(max_iter=2000).fit(train_scores[score_columns], train_scores["label"])
    predicted = classifier.predict(test_scores[score_columns])

    predictions = pd.DataFrame({"batch": folder.name, "method": "MEFISTO",
                                "subject_id": test_scores["subject_id"], "truth": test_scores["label"],
                                "predicted": predicted})
    metrics = pd.DataFrame([{"batch": folder.name, "method": "MEFISTO", "status": "success",
                             "accuracy": accuracy_score(test_scores["label"], predicted),
                             "balanced_accuracy": balanced_accuracy_score(test_scores["label"], predicted),
                             "macro_f1": f1_score(test_scores["label"], predicted, average="macro"),
                             "elapsed_seconds": time.perf_counter() - started, "error": ""}])

    train_samples.to_csv(batch_output / "train_sample_factors.csv.gz", index=False)
    test_samples.to_csv(batch_output / "test_sample_factors.csv.gz", index=False)
    train_scores.to_csv(batch_output / "train_subject_scores.csv.gz", index=False)
    test_scores.to_csv(batch_output / "test_subject_scores.csv.gz", index=False)
    pd.DataFrame(weights, index=train.columns, columns=names).rename_axis("feature_id").reset_index().to_csv(
        batch_output / "feature_loadings.csv.gz", index=False)
    predictions.to_csv(batch_output / "predictions.csv.gz", index=False)
    metrics.to_csv(batch_output / "metrics.csv", index=False)
    return metrics, predictions

results = [run_batch(folder, i) for i, folder in enumerate(batch_folders, 1)]
all_metrics = pd.concat([x[0] for x in results], ignore_index=True)
all_predictions = pd.concat([x[1] for x in results], ignore_index=True)
all_metrics.to_csv(output_folder / "all_metrics.csv", index=False)
all_predictions.to_csv(output_folder / "all_predictions.csv.gz", index=False)
print("Saved:", output_folder)
all_metrics



        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Loaded view='data' group='0' with N=8 samples and D=32 features...
Loaded view='data' group='1' with N=9 samples and D=32 features...
Loaded view='data' group='2' with N=3 samples and D=32 features...
Loaded view='data' group='3' with N=8 samples and D=32 features...
Loaded view='data' group='4' with N=9 samples and D=32 features...
Loaded view='data' group='5' with N=7 samples and D=3